<a href="https://colab.research.google.com/github/MaggieHDez/ClassFiles/blob/IDA/practica_py_spark_255879.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Actividad práctica: RDD en PySpark (Creación, Transformaciones y Acciones)
>Clase: Ingeniería de Datos Avanzada\
>Alumno: Margarita Cristina Hernández Delgadillo\
>Matrícula: 255879\
>Fecha: 24 de Marzo de 2026

##1. Instalación / Verificación de `PySpark`

In [34]:
# Verificamos si PySpark ya está instalado en el entorno
try:
    import pyspark
    print("PySpark ya está instalado")
    print("Versión:", pyspark.__version__)
except ModuleNotFoundError:
    # Si no está instalado, lo instalamos
    print("PySpark no está instalado. Instalando...")
    !pip install pyspark -q
    import pyspark
    print("Instalación completada")
    print("Versión:", pyspark.__version__)

PySpark ya está instalado
Versión: 4.0.2


##2. Creación de `SparkSession` y `SparkContext`

In [35]:
# Creamos SparkSession, el punto de entrada principal
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Actividad_RDD_IDA") \
    .getOrCreate()

# Creamos SparkContext para trabajar directamente con RDDs
sc = spark.sparkContext

print("SparkSession y SparkContext creados correctamente\n")
print(f"  - Nombre de la aplicación: {sc.appName}")
print(f"  - Master: {sc.master}")
print(f"  - Versión de Spark: {sc.version}")
print(f"  - Python Version: {sc.pythonVer}")
print(f"  - Default Parallelism: {sc.defaultParallelism}") # Nivel de paralelismo por defecto, cuántas particiones (tareas paralelas) se usarán

SparkSession y SparkContext creados correctamente

  - Nombre de la aplicación: Actividad_RDD_IDA
  - Master: local[*]
  - Versión de Spark: 4.0.2
  - Python Version: 3.12
  - Default Parallelism: 2


##3. Generación de datos

In [36]:
# Importación de numpy para generar datos aleatorios
import numpy as np

# Fijamos la semilla para que los resultados sean reproducibles
np.random.seed(18)

# Definición de los parámetros de la distribución
media = 50
desviacion = 10
n = 1000  # tamaño de la muestra

# Generación de los datos usando la distribución normal
datos = np.random.normal(loc=media, scale=desviacion, size=n)

# Conversión a lista (se redondean los resultados para mejor interpretación)
lista_datos = [round(x, 2) for x in datos]

# Imprimimos para comprobar
print("Primeros 10 valores generados:")
print(lista_datos[:10])
print("Total de valores:", len(lista_datos))

Primeros 10 valores generados:
[np.float64(50.79), np.float64(71.9), np.float64(48.65), np.float64(51.61), np.float64(54.43), np.float64(56.23), np.float64(60.09), np.float64(53.94), np.float64(46.64), np.float64(43.54)]
Total de valores: 1000


##4. Creación del `rdd` principal

In [37]:
# Convertimos la lista de Python en un RDD
rdd0 = sc.parallelize(lista_datos)

print("RDD creado correctamente:")
print(rdd0)

# count() devuelve el número de elementos
print("Cantidad de elementos en rdd0:", rdd0.count())

RDD creado correctamente:
ParallelCollectionRDD[185] at readRDDFromFile at PythonRDD.scala:297
Cantidad de elementos en rdd0: 1000


##5. Transformaciones sobre el `rdd`

In [38]:
# map(): transforma cada elemento del RDD aplicando una función
# En este caso, elevamos cada valor al cuadrado
rdd0_cuadrado = rdd0.map(lambda x: x ** 2)

# take(n): Muestra los primeros n elementos en este caso n = 5
# Mostramos los primeros 5 elementos de rdd0 antes de elevar al cuadrado
print("5 elementos de rdd0:")
print(rdd0.take(5))

# Mostramos los primeros 5 elementos del resultado de elevar al cuadrado
print("\n5 elementos de rdd_cuadrado:")
print(rdd0_cuadrado.take(5))

# filter(): selecciona solo los elementos que cumplen una condición
# Filtramos valores mayores a 50 en rdd0
rdd0_mayores_50 = rdd0.filter(lambda x: x > 50)

# Mostramos los primeros 4 elementos del resultado de filtrar
print("\n4 elementos mayores a 50:")
print(rdd0_mayores_50.take(4))

# Contar cuántos cumplen la condición en rdd0
print("\nCantidad de valores mayores a 50:")
print(rdd0_mayores_50.count())

5 elementos de rdd0:
[np.float64(50.79), np.float64(71.9), np.float64(48.65), np.float64(51.61), np.float64(54.43)]

5 elementos de rdd_cuadrado:
[np.float64(2579.6241), np.float64(5169.610000000001), np.float64(2366.8224999999998), np.float64(2663.5921), np.float64(2962.6249)]

4 elementos mayores a 50:
[np.float64(50.79), np.float64(71.9), np.float64(51.61), np.float64(54.43)]

Cantidad de valores mayores a 50:
502


##6. Generación de pseudo-conjuntos

En esta sección se generan dos RDD independientes con la misma distribución normal,
pero utilizando diferentes semillas para simular muestras distintas de la misma población.
Esto permite aplicar correctamente las transformaciones de pseudo-conjuntos.

In [39]:
# Generamos el primer conjunto
np.random.seed(18)
lista1 = [round(x, 2) for x in np.random.normal(media, desviacion, n)]

# Generamos un segundo conjunto (cambiando la semilla para no obtener los mismos resultados)
np.random.seed(25)
lista2 = [round(x, 2) for x in np.random.normal(media, desviacion, n)]

# Convertimos ambas listas a RDD
rdd1 = sc.parallelize(lista1)
rdd2 = sc.parallelize(lista2)

print("Tamaño de rdd1:", rdd1.count())
print("5 elementos de rdd1:")
print(rdd1.take(5))
print("\nTamaño de rdd2:", rdd2.count())
print("5 elementos de rdd2:")
print(rdd2.take(5))

Tamaño de rdd1: 1000
5 elementos de rdd1:
[np.float64(50.79), np.float64(71.9), np.float64(48.65), np.float64(51.61), np.float64(54.43)]

Tamaño de rdd2: 1000
5 elementos de rdd2:
[np.float64(52.28), np.float64(60.27), np.float64(41.6), np.float64(44.09), np.float64(40.43)]


##7. Transformaciones de pseudo-conjuntos y visualización de resultados

In [40]:
# union(): combina los elementos de ambos RDD
rdd_union = rdd1.union(rdd2)

# distinct(): elimina elementos duplicados
rdd_distinct = rdd_union.distinct()

# intersection(): obtiene elementos comunes entre ambos RDD
rdd_intersection = rdd1.intersection(rdd2)

# subtract(): elementos que están en rdd1 pero no en rdd2
rdd_subtract = rdd1.subtract(rdd2)

# cartesian(): esta operación genera todas las combinaciones posibles entre ambos RDD (n x n)
# NOTA: Debido a la combinacion, puede crecer bastante (1000 x 1000 = 1,000,000 combinaciones)
rdd_cartesian = rdd1.cartesian(rdd2)

# visualización de resultados
print("=== UNION ===")
print("Primeros 5 elementos:", rdd_union.take(5))
print("5 elementos más pequeños ordenados:", rdd_union.takeOrdered(5))
print("Elementos totales:", rdd_union.count())

print("\n=== DISTINCT ===")
print("Primeros 5 elementos:", rdd_distinct.take(5))
print("5 elementos más pequeños ordenados:", rdd_distinct.takeOrdered(5))
print("Elementos totales:", rdd_distinct.count())

print("\n=== INTERSECTION ===")
print("Primeros 5 elementos:", rdd_intersection.take(5))
print("5 elementos más pequeños ordenados:", rdd_intersection.takeOrdered(5))
print("Elementos totales:", rdd_intersection.count())

print("\n=== SUBTRACT ===")
print("Primeros 5 elementos:", rdd_subtract.take(5))
print("5 elementos más pequeños ordenados:", rdd_subtract.takeOrdered(5))
print("Elementos totales:", rdd_subtract.count())

print("\n=== CARTESIAN ===")
print("Primeros 5 elementos:", rdd_cartesian.take(5))
print("5 elementos más pequeños ordenados:", rdd_cartesian.takeOrdered(5))
print("Elementos totales:", rdd_cartesian.count())

=== UNION ===
Primeros 5 elementos: [np.float64(50.79), np.float64(71.9), np.float64(48.65), np.float64(51.61), np.float64(54.43)]
5 elementos más pequeños ordenados: [np.float64(16.82), np.float64(17.92), np.float64(18.86), np.float64(19.23), np.float64(19.5)]
Elementos totales: 2000

=== DISTINCT ===
Primeros 5 elementos: [np.float64(48.65), np.float64(56.23), np.float64(60.09), np.float64(40.05), np.float64(56.01)]
5 elementos más pequeños ordenados: [np.float64(16.82), np.float64(17.92), np.float64(18.86), np.float64(19.23), np.float64(19.5)]
Elementos totales: 1566

=== INTERSECTION ===
Primeros 5 elementos: [np.float64(60.09), np.float64(56.77), np.float64(44.44), np.float64(44.58), np.float64(60.19)]
5 elementos más pequeños ordenados: [np.float64(30.68), np.float64(36.0), np.float64(36.16), np.float64(36.89), np.float64(37.3)]
Elementos totales: 191

=== SUBTRACT ===
Primeros 5 elementos: [np.float64(48.65), np.float64(48.65), np.float64(56.23), np.float64(40.05), np.float64(56

##8. Operaciones estadísticas y matemáticas sobre `rdd0`

In [41]:
# Importamos math para realizar los cálculos de raíz cuadrada
import math

# Aplicamos raíz cuadrada a cada elemento usando map()
rdd0_raiz = rdd0.map(lambda x: round(math.sqrt(x), 4))

# mean(): calculamos los promedio de rdd0
promedio = rdd0.mean()

# max() y min(): calculamos los valores extremos de rdd0
maximo = rdd0.max()
minimo = rdd0.min()

# variance() y stdev(): calculamos la dispersión de los datos de rdd0
varianza = rdd0.variance()
desv_std = rdd0.stdev()

# visualización
print("=== ESTADÍSTICAS ===")
print(f"Promedio: {promedio:.4f}")
print(f"Máximo: {maximo:.4f}")
print(f"Mínimo: {minimo:.4f}")
print(f"Varianza: {varianza:.4f}")
print(f"Desviación estándar: {desv_std:.4f}")
print("\n=== MATEMÁTICAS ===")
# Mostramos solo 5 resultados de raíz cuadrada
print(f"5 elementos de raíz cuadrada: {rdd0_raiz.take(5)}")


=== ESTADÍSTICAS ===
Promedio: 49.9198
Máximo: 89.4700
Mínimo: 16.8200
Varianza: 107.7813
Desviación estándar: 10.3818

=== MATEMÁTICAS ===
5 elementos de raíz cuadrada: [7.1267, 8.4794, 6.975, 7.184, 7.3777]


##9. Conclusión

En esta práctica se trabajó con RDDs en PySpark para aplicar transformaciones,
acciones y operaciones de pseudo-conjuntos. Se generaron datos con distribución normal,
lo que permitió analizar su comportamiento mediante estadísticas como promedio,
varianza y desviación estándar. Además, se observó el uso de operaciones como union,
intersection y cartesian, destacando el impacto del tamaño de los datos en el procesamiento.